In [1]:
# ============================================================
# Célula 1 — Funções de limpeza
# ============================================================
import numpy as np
import pandas as pd


def tirar_notas_de_rodape(texto):
    # apaga as notas da Wikipedia, que ficam entre colchetes
    # exemplo: "$821,708,551[33]" vira "$821,708,551"
    while "[" in texto and "]" in texto:
        inicio = texto.find("[")
        fim = texto.find("]")
        texto = texto[:inicio] + texto[fim + 1:]
    return texto


def limpar_texto(valor):
    # célula vazia vira texto vazio
    if pd.isna(valor):
        return ""
    texto = str(valor)
    texto = tirar_notas_de_rodape(texto)
    # a Wikipedia coloca um caractere invisível no meio de "Budget",
    # aqui eu apago ele
    texto = texto.replace("\xad", "")
    texto = texto.strip()   # tira espaços no começo e no fim
    return texto


def texto_para_numero(valor):
    # exemplos: "$821,708,551" vira 821708551
    #           "90% (249 reviews)" vira 90
    #           "—N/a" vira vazio
    texto = limpar_texto(valor)
    texto = texto.replace("$", "")
    texto = texto.replace(",", "")
    texto = texto.replace("%", "")

    # fico só com a primeira palavra: "90 (249 reviews)" vira "90"
    palavras = texto.split(" ")
    primeira = palavras[0]

    # se não der para virar número (ex: "—N/a"), fica vazio
    try:
        return float(primeira)
    except ValueError:
        return np.nan


def orcamento_para_dolares(valor):
    # o orçamento vem em milhões: "$200" quer dizer 200 milhões
    # alguns vêm em faixa, ex: "$75–83"; nesse caso uso a média (79 milhões)
    texto = limpar_texto(valor)
    texto = texto.replace("$", "")
    texto = texto.replace(",", "")

    try:
        if "–" in texto:
            partes = texto.split("–")
            menor = float(partes[0])
            maior = float(partes[1])
            milhoes = (menor + maior) / 2
        else:
            milhoes = float(texto)
    except ValueError:
        return np.nan

    return milhoes * 1000000


def separar_titulo_ano(titulo):
    # na tabela de críticas, os títulos repetidos vêm com o ano
    # exemplo: "Batman (1966)" vira "Batman" e 1966
    #          "Joker" continua "Joker" e o ano fica vazio
    titulo = limpar_texto(titulo)
    final = titulo[-6:]   # os 6 últimos caracteres, ex: "(1966)"

    if final.startswith("(") and final.endswith(")"):
        ano = final[1:5]   # o que está entre os parênteses
        if ano.isdigit():
            titulo_sem_ano = titulo[:-6].strip()
            return titulo_sem_ano, int(ano)

    return titulo, np.nan


# nome da coluna na Wikipedia (já sem as notas [33]) -> nome novo
NOVOS_NOMES = {
    "Title": "titulo",
    "Film": "titulo",
    "Distributor(s)": "distribuidora",
    "Release date (United States)": "data_lancamento",
    "Budget (millions)": "orcamento",
    "Box office gross Opening weekend (North America)": "bilheteria_estreia",
    "Box office gross North America": "bilheteria_america_norte",
    "Box office gross Other territories": "bilheteria_outros",
    "Box office gross Worldwide": "bilheteria_mundial",
    "Rotten Tomatoes": "rotten_tomatoes",
    "Critical Rotten Tomatoes": "rotten_tomatoes",
    "Metacritic": "metacritic",
    "Critical Metacritic": "metacritic",
}


def padronizar_colunas(tabela):
    # limpa os nomes das colunas e troca pelos nomes em português
    nomes_limpos = []
    for coluna in tabela.columns:
        nomes_limpos.append(limpar_texto(coluna))
    tabela.columns = nomes_limpos
    tabela = tabela.rename(columns=NOVOS_NOMES)
    return tabela

In [2]:
# ============================================================
# Célula 2 — Limpar cada tabela e juntar bilheteria com críticas
# ============================================================

def limpar_bilheteria(caminho):
    tabela = pd.read_csv(caminho)
    tabela = padronizar_colunas(tabela)
    tabela["titulo"] = tabela["titulo"].apply(limpar_texto)

    # as últimas linhas são "Total" e "Average", que não são filmes
    tabela = tabela[tabela["titulo"] != "Total"]
    tabela = tabela[tabela["titulo"] != "Average"]
    tabela = tabela.copy()

    # data de lançamento e ano
    # o formato da data é tipo "July 30, 1966"
    datas = tabela["data_lancamento"].apply(limpar_texto)
    tabela["data_lancamento"] = pd.to_datetime(datas, format="%B %d, %Y")
    tabela["ano"] = tabela["data_lancamento"].dt.year

    # orçamento: texto -> dólares
    tabela["orcamento"] = tabela["orcamento"].apply(orcamento_para_dolares)

    # a DC não tem as colunas "distribuidora" e "bilheteria_outros",
    # então crio elas vazias para a tabela ficar igual à da Marvel
    if "distribuidora" not in tabela.columns:
        tabela["distribuidora"] = np.nan
    if "bilheteria_outros" not in tabela.columns:
        tabela["bilheteria_outros"] = np.nan

    # bilheterias: texto -> número
    colunas_bilheteria = ["bilheteria_estreia", "bilheteria_america_norte",
                          "bilheteria_outros", "bilheteria_mundial"]
    for coluna in colunas_bilheteria:
        tabela[coluna] = tabela[coluna].apply(texto_para_numero)

    # fico só com as colunas que vou usar
    colunas = ["titulo", "ano", "data_lancamento", "distribuidora", "orcamento",
               "bilheteria_estreia", "bilheteria_america_norte",
               "bilheteria_outros", "bilheteria_mundial"]
    return tabela[colunas]


def limpar_criticas(caminho):
    tabela = pd.read_csv(caminho)
    tabela = padronizar_colunas(tabela)

    # separo o título e o ano: "Batman (1966)" -> "Batman" e 1966
    titulos = []
    anos = []
    for titulo in tabela["titulo"]:
        titulo_limpo, ano = separar_titulo_ano(titulo)
        titulos.append(titulo_limpo)
        anos.append(ano)
    tabela["titulo"] = titulos
    tabela["ano_critica"] = anos

    # notas: texto -> número
    tabela["rotten_tomatoes"] = tabela["rotten_tomatoes"].apply(texto_para_numero)
    tabela["metacritic"] = tabela["metacritic"].apply(texto_para_numero)

    return tabela[["titulo", "ano_critica", "rotten_tomatoes", "metacritic"]]


def juntar(bilheteria, criticas, nome):
    # junto as duas tabelas pelo título
    juntas = bilheteria.merge(criticas, on="titulo", how="left")

    # problema: títulos repetidos (ex: Batman de 1966 e de 1989) se misturam,
    # o Batman de 1966 ganha uma linha com a crítica do de 1989 e vice-versa.
    # então fico só com as linhas em que a crítica não tem ano
    # ou em que o ano da crítica é igual ao ano do filme
    linhas_certas = []
    for i, linha in juntas.iterrows():
        if pd.isna(linha["ano_critica"]) or linha["ano_critica"] == linha["ano"]:
            linhas_certas.append(i)
    juntas = juntas.loc[linhas_certas]
    juntas = juntas.drop(columns="ano_critica")

    # conferências
    print("=" * 50)
    print(nome)
    print("=" * 50)
    print("Filmes na bilheteria:   ", len(bilheteria))
    print("Filmes depois de juntar:", len(juntas))

    # algum filme sumiu na junção?
    sumiram = []
    for titulo in bilheteria["titulo"]:
        if titulo not in list(juntas["titulo"]):
            sumiram.append(titulo)
    print("Sumiram na junção:      ", sumiram)

    # algum filme apareceu duas vezes?
    repetidos = juntas[juntas.duplicated(["titulo", "ano"])]
    print("Linhas repetidas:       ", list(repetidos["titulo"]))

    # algum filme ficou sem nenhuma nota de crítica?
    sem_critica = []
    for i, linha in juntas.iterrows():
        if pd.isna(linha["rotten_tomatoes"]) and pd.isna(linha["metacritic"]):
            sem_critica.append(linha["titulo"])
    print("Sem nota de crítica:    ", sem_critica)
    print()

    return juntas


marvel = juntar(limpar_bilheteria("../data/raw/marvel_bilheteria_wiki.csv"),
                limpar_criticas("../data/raw/marvel_criticas_wiki.csv"),
                "MARVEL")

dc = juntar(limpar_bilheteria("../data/raw/dc_bilheteria_wiki.csv"),
            limpar_criticas("../data/raw/dc_criticas_wiki.csv"),
            "DC")

MARVEL
Filmes na bilheteria:    80
Filmes depois de juntar: 80
Sumiram na junção:       []
Linhas repetidas:        []
Sem nota de crítica:     ['Inhumans']

DC
Filmes na bilheteria:    51
Filmes depois de juntar: 51
Sumiram na junção:       []
Linhas repetidas:        []
Sem nota de crítica:     ['Aztec Batman: Clash of Empires']



In [3]:
# ============================================================
# Célula 3 — Exclusões, formato e consistência
# ============================================================

# --- 1. Filmes que saem e o motivo (vai para o README) ---
EXCLUIR = {
    "Inhumans": "série de TV",
    "Constantine": "selo Vertigo",
    "Batman: The Killing Joke": "direto em vídeo",
    "Batman: Return of the Caped Crusaders": "direto em vídeo",
    "Batman and Harley Quinn": "direto em vídeo",
}
# Obs.: Kick-Ass, Kingsman (selo Icon) e Men in Black (selo Malibu)
# não aparecem na página da Marvel, então não precisam estar aqui.

# Critério de bilheteria completa: só entram filmes que já saíram do cinema.
# Os dados não dizem quando o filme saiu de cartaz, então uso uma regra:
# o filme precisa ter estreado pelo menos DIAS_EM_CARTAZ dias antes da coleta.
DATA_COLETA = pd.Timestamp("2026-11-02")   # dia em que as tabelas da Wikipedia foram baixadas
DIAS_EM_CARTAZ = 120                       # cerca de 4 meses, tempo típico de um filme em cartaz

# --- 2. Animações de cinema (o resto é live-action) ---
ANIMACOES = [
    "Big Hero 6",
    "Spider-Man: Into the Spider-Verse",
    "Spider-Man: Across the Spider-Verse",
    "Batman: Mask of the Phantasm",
    "The Lego Batman Movie",
    "Teen Titans Go! To the Movies",
    "DC League of Super-Pets",
    "Aztec Batman: Clash of Empires",
]

# --- 3. Juntar Marvel e DC numa tabela só ---
marvel["universo"] = "Marvel"
dc["universo"] = "DC"
filmes = pd.concat([marvel, dc], ignore_index=True)
total_antes = len(filmes)

# --- 4. Tirar os excluídos ---
print("=" * 50)
print("FILMES EXCLUÍDOS")
print("=" * 50)

linhas_para_tirar = []
for i, filme in filmes.iterrows():
    if filme["titulo"] in EXCLUIR:
        motivo = EXCLUIR[filme["titulo"]]
        print(f"• {filme['titulo']} ({filme['ano']}, {filme['universo']}): {motivo}")
        linhas_para_tirar.append(i)

    # filmes que ainda podem estar em cartaz (bilheteria incompleta)
    elif (DATA_COLETA - filme["data_lancamento"]).days < DIAS_EM_CARTAZ:
        dias = (DATA_COLETA - filme["data_lancamento"]).days
        print(f"• {filme['titulo']} ({filme['ano']}, {filme['universo']}): "
              f"estreou há {dias} dias, pode ainda estar em cartaz")
        linhas_para_tirar.append(i)

    # Batman (1966): tiro pelo título E pelo ano, porque o Batman de 1989 fica
    if filme["titulo"] == "Batman" and filme["ano"] == 1966:
        print("• Batman (1966, DC): bilheteria registrada não confiável (US$ 3.315)")
        linhas_para_tirar.append(i)

filmes = filmes.drop(linhas_para_tirar)

# --- 5. Coluna formato ---
formatos = []
for titulo in filmes["titulo"]:
    if titulo in ANIMACOES:
        formatos.append("animação")
    else:
        formatos.append("live-action")
filmes["formato"] = formatos

print()
print("=" * 50)
print("ANIMAÇÕES MARCADAS")
print("=" * 50)
for i, filme in filmes.iterrows():
    if filme["formato"] == "animação":
        print(f"• {filme['titulo']} ({filme['ano']})")

# --- 6. Consistência: América do Norte + outros = mundial? (só Marvel) ---
print()
print("=" * 50)
print("CONSISTÊNCIA (Marvel)")
print("=" * 50)

problemas = 0
for i, filme in filmes.iterrows():
    if filme["universo"] == "Marvel":
        soma = filme["bilheteria_america_norte"] + filme["bilheteria_outros"]
        diferenca = abs(soma - filme["bilheteria_mundial"])
        if diferenca > 1000:
            problemas = problemas + 1
            print(f"• {filme['titulo']}: diferença de US$ {diferenca:,.0f}")

if problemas == 0:
    print("Todos os filmes batem.")

# Na DC, "outros territórios" está vazio. Calculo: mundial − América do Norte
# (fillna preenche só os valores vazios, então a Marvel não muda)
outros_calculado = filmes["bilheteria_mundial"] - filmes["bilheteria_america_norte"]
# se o resultado for 0, a Wikipedia só tem a bilheteria da América do Norte
# (ex: Supergirl 1984). Isso não quer dizer que o filme faturou zero fora,
# e sim que o dado não existe, então fica vazio (NaN)
outros_calculado = outros_calculado.replace(0, np.nan)
filmes["bilheteria_outros"] = filmes["bilheteria_outros"].fillna(outros_calculado)

# --- 7. Resumo ---
quantos_marvel = len(filmes[filmes["universo"] == "Marvel"])
quantos_dc = len(filmes[filmes["universo"] == "DC"])

print()
print("=" * 50)
print("RESUMO")
print("=" * 50)
print("Filmes antes das exclusões: ", total_antes)
print("Filmes depois das exclusões:", len(filmes))
print("  Marvel:", quantos_marvel)
print("  DC:    ", quantos_dc)

FILMES EXCLUÍDOS
• Inhumans (2017, Marvel): série de TV
• Spider-Man: Brand New Day (2026, Marvel): estreou há 94 dias, pode ainda estar em cartaz
• Batman (1966, DC): bilheteria registrada não confiável (US$ 3.315)
• Constantine (2005, DC): selo Vertigo
• Batman: The Killing Joke (2016, DC): direto em vídeo
• Batman: Return of the Caped Crusaders (2016, DC): direto em vídeo
• Batman and Harley Quinn (2017, DC): direto em vídeo
• Clayface (2026, DC): estreou há 10 dias, pode ainda estar em cartaz

ANIMAÇÕES MARCADAS
• Big Hero 6 (2014)
• Spider-Man: Into the Spider-Verse (2018)
• Spider-Man: Across the Spider-Verse (2023)
• Batman: Mask of the Phantasm (1993)
• The Lego Batman Movie (2017)
• Teen Titans Go! To the Movies (2018)
• DC League of Super-Pets (2022)
• Aztec Batman: Clash of Empires (2025)

CONSISTÊNCIA (Marvel)
• The Wolverine: diferença de US$ 5,892

RESUMO
Filmes antes das exclusões:  131
Filmes depois das exclusões: 123
  Marvel: 78
  DC:     45


In [4]:
# ============================================================
# Célula 4 — Consultar a OMDb (nota do IMDb e número de votos)
# ============================================================
import os
import time
import requests
from dotenv import load_dotenv

# --- 1. Ler a chave do arquivo .env (nunca escrever a chave aqui) ---
load_dotenv("../.env")
CHAVE = os.getenv("OMDB_KEY")
print("Chave encontrada:", CHAVE is not None)


# --- 2. Função que busca um filme ---
def buscar_omdb(titulo, ano):
    # devolve os dados do filme, ou None se a OMDb não encontrar
    parametros = {"apikey": CHAVE, "t": titulo, "y": ano}
    resposta = requests.get("https://www.omdbapi.com/", params=parametros)
    dados = resposta.json()
    if dados["Response"] == "False":
        return None
    return dados


# --- 3. Buscar todos os filmes ---
# se o omdb.csv já existe, não precisa consultar de novo
# (o limite da chave é de 1.000 consultas por dia)
if os.path.exists("../data/raw/omdb.csv"):
    print("O arquivo omdb.csv já existe, a consulta foi pulada.")
else:
    resultados = []
    nao_encontrados = []

    for i, filme in filmes.iterrows():
        dados = buscar_omdb(filme["titulo"], int(filme["ano"]))
        if dados is None:
            nao_encontrados.append(f"{filme['titulo']} ({filme['ano']})")
        else:
            resultados.append({
                "titulo": filme["titulo"],
                "ano": filme["ano"],
                "titulo_omdb": dados["Title"],
                "ano_omdb": dados["Year"],
                "imdb_id": dados["imdbID"],
                "nota_imdb": dados["imdbRating"],
                "votos_imdb": dados["imdbVotes"],
                "bilheteria_eua_omdb": dados["BoxOffice"],
            })
        time.sleep(0.2)   # pausa curta para não sobrecarregar a API

    # --- 4. Salvar o bruto ---
    omdb = pd.DataFrame(resultados)
    omdb.to_csv("../data/raw/omdb.csv", index=False)

    print("Filmes consultados:", len(filmes))
    print("Encontrados:       ", len(resultados))
    print("Não encontrados:   ", nao_encontrados)

# --- 5. Conferir os títulos ---
# a OMDb pode devolver um filme diferente com nome parecido
omdb = pd.read_csv("../data/raw/omdb.csv")

print()
print("=" * 50)
print("TÍTULO DA OMDb DIFERENTE DO NOSSO")
print("=" * 50)
for i, linha in omdb.iterrows():
    if linha["titulo"] != linha["titulo_omdb"]:
        print(f"• {linha['titulo']}  →  {linha['titulo_omdb']}")

Chave encontrada: True
O arquivo omdb.csv já existe, a consulta foi pulada.

TÍTULO DA OMDb DIFERENTE DO NOSSO
• X2  →  X2: X-Men United
• Guardians of the Galaxy Vol. 2  →  Guardians of the Galaxy: Vol. 2
• Dark Phoenix  →  X-Men: Dark Phoenix
• Spider-Man: Far From Home  →  Spider-Man: Far from Home
• Teen Titans Go! To the Movies  →  Teen Titans GO! To the Movies
• Birds of Prey  →  Birds of Prey and the Fantabulous Emancipation of One Harley Quinn


In [5]:
# ============================================================
# Célula 5 — Juntar a OMDb e aplicar o critério de dados suficientes
# ============================================================

# --- 1. Converter os textos da OMDb em números ---
omdb["nota_imdb"] = omdb["nota_imdb"].apply(texto_para_numero)
omdb["votos_imdb"] = omdb["votos_imdb"].apply(texto_para_numero)
omdb["bilheteria_eua_omdb"] = omdb["bilheteria_eua_omdb"].apply(texto_para_numero)
omdb = omdb[["titulo", "ano", "imdb_id", "nota_imdb", "votos_imdb", "bilheteria_eua_omdb"]]

# --- 2. Juntar com a tabela de filmes (por título + ano) ---
filmes = filmes.merge(omdb, on=["titulo", "ano"], how="left")
total_antes = len(filmes)

# --- 3. Tirar os filmes com dados insuficientes ---
print("=" * 50)
print("SAÍRAM POR FALTA DE DADOS")
print("=" * 50)

linhas_para_tirar = []
for i, filme in filmes.iterrows():
    falta = []
    if pd.isna(filme["bilheteria_mundial"]):
        falta.append("bilheteria mundial")
    if pd.isna(filme["orcamento"]):
        falta.append("orçamento")
    if pd.isna(filme["nota_imdb"]):
        falta.append("nota do IMDb")
    if pd.isna(filme["rotten_tomatoes"]) and pd.isna(filme["metacritic"]):
        falta.append("crítica")

    if len(falta) > 0:
        texto_falta = ", ".join(falta)   # junta a lista num texto só
        print(f"• {filme['titulo']} ({filme['ano']}, {filme['universo']}): falta {texto_falta}")
        linhas_para_tirar.append(i)

filmes = filmes.drop(linhas_para_tirar)

# --- 4. Resumo ---
quantos_marvel = len(filmes[filmes["universo"] == "Marvel"])
quantos_dc = len(filmes[filmes["universo"] == "DC"])

print()
print("=" * 50)
print("RESUMO")
print("=" * 50)
print("Filmes antes do filtro: ", total_antes)
print("Filmes depois do filtro:", len(filmes))
print("  Marvel:", quantos_marvel)
print("  DC:    ", quantos_dc)


SAÍRAM POR FALTA DE DADOS
• Swamp Thing (1982, DC): falta bilheteria mundial
• The Return of Swamp Thing (1989, DC): falta orçamento
• Aztec Batman: Clash of Empires (2025, DC): falta orçamento, crítica

RESUMO
Filmes antes do filtro:  123
Filmes depois do filtro: 120
  Marvel: 78
  DC:     42


In [6]:
# ============================================================
# Célula 6 — Lucro, ROI, conferência com a OMDb e CSV final
# ============================================================

# --- 1. Lucro e ROI ---
filmes["lucro"] = filmes["bilheteria_mundial"] - filmes["orcamento"]
# ROI clássico: (receita - custo) / custo
# exemplo: ROI 2 = o filme lucrou 2 vezes o que custou; ROI negativo = prejuízo
filmes["roi"] = (filmes["bilheteria_mundial"] - filmes["orcamento"]) / filmes["orcamento"]

# --- 2. Conferência: bilheteria EUA (OMDb) x América do Norte (Wikipedia) ---
# as duas deveriam ser parecidas; diferença acima de 5% merece uma olhada
print("=" * 50)
print("CONFERÊNCIA: OMDb x Wikipedia (diferença acima de 5%)")
print("=" * 50)

problemas = 0
for i, filme in filmes.iterrows():
    wiki = filme["bilheteria_america_norte"]
    omdb_eua = filme["bilheteria_eua_omdb"]
    if pd.isna(wiki) or pd.isna(omdb_eua):
        continue   # pula os filmes sem um dos dois valores
    diferenca = abs(wiki - omdb_eua) / wiki * 100
    if diferenca > 5:
        problemas = problemas + 1
        print(f"• {filme['titulo']} ({filme['ano']}): diferença de {diferenca:.1f}%")

if problemas == 0:
    print("Todos os filmes batem (diferença de até 5%).")

# --- 3. Organizar as colunas e salvar ---
colunas_finais = ["titulo", "ano", "universo", "formato", "data_lancamento",
                  "distribuidora", "orcamento", "bilheteria_estreia",
                  "bilheteria_america_norte", "bilheteria_outros",
                  "bilheteria_mundial", "lucro", "roi",
                  "rotten_tomatoes", "metacritic", "nota_imdb", "votos_imdb",
                  "imdb_id"]
filmes = filmes[colunas_finais]
filmes = filmes.sort_values(["universo", "ano"])
filmes.to_csv("../data/processed/filmes_dc_marvel.csv", index=False)

# filme com maior lucro e maior ROI (ordeno do maior para o menor e pego o primeiro)
maior_lucro = filmes.sort_values("lucro", ascending=False).iloc[0]
maior_roi = filmes.sort_values("roi", ascending=False).iloc[0]

print()
print("=" * 50)
print("CSV FINAL")
print("=" * 50)
print("Salvo em: data/processed/filmes_dc_marvel.csv")
print("Linhas:  ", len(filmes))
print("Colunas: ", len(filmes.columns))
print()
print("Maior lucro:", maior_lucro["titulo"])
print("Maior ROI:  ", maior_roi["titulo"])

CONFERÊNCIA: OMDb x Wikipedia (diferença acima de 5%)
Todos os filmes batem (diferença de até 5%).

CSV FINAL
Salvo em: data/processed/filmes_dc_marvel.csv
Linhas:   120
Colunas:  18

Maior lucro: Avengers: Endgame
Maior ROI:   Joker
